# RCDM Training on Google Colab

This notebook runs your RCDM (Representation-Conditioned Diffusion Model) training on Google Colab's free GPU.

## Setup Steps:
1. **Runtime → Change runtime type → GPU (T4 recommended)**
2. Run cells in order
3. Upload your data or connect to Google Drive

## 1. Check GPU Availability

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
Memory: 85.02 GB


## 2. Mount Google Drive (Optional - for storing checkpoints)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
!mkdir -p /content/drive/MyDrive/rcdm_checkpoints

Mounted at /content/drive


## 3. Clone/Upload Your Code

**Option A: Upload as ZIP** (recommended if code isn't on GitHub)

In [ ]:
from google.colab import files
import zipfile
import os

# Upload your master_implementation.zip
print("Upload your zipped project folder...")
uploaded = files.upload()

# Extract into /content/master_implementation/ so paths are consistent
os.makedirs('/content/master_implementation', exist_ok=True)
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/master_implementation')
        print(f"Extracted {filename}")

# Verify structure
print("\nProject structure:")
for item in sorted(os.listdir('/content/master_implementation')):
    print(f"  {item}/")

# Navigate to project root
%cd /content/master_implementation

Upload your zipped project folder...


**Option B: Clone from GitHub** (if your code is in a repo)

In [ ]:
# Uncomment and modify if using GitHub:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

## 4. Install Dependencies

In [4]:
# Install guided_diffusion package (setup.py lives at guided_diffusion/setup.py)
!pip install -e /content/master_implementation/guided_diffusion -q

# Install other dependencies
!pip install "blobfile>=1.0.5" tqdm pillow torchvision -q

print("✓ Dependencies installed")

ERROR: ./guided_diffusion is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
✓ Dependencies installed


## 5. Upload Training Data

You need to upload `train_reps.pt` (826MB) to Colab or Google Drive.

In [ ]:
# Option A: Upload train_reps.pt directly
!mkdir -p /content/master_implementation/data/tiny-imagenet-200

print("Upload your train_reps.pt file (this may take a few minutes - 826MB)...")
uploaded = files.upload()

# Move to expected location
!mv train_reps.pt /content/master_implementation/data/tiny-imagenet-200/

print("✓ Data uploaded")

In [ ]:
# Option B: Copy from Google Drive (if you uploaded it there beforehand)
# !cp /content/drive/MyDrive/train_reps.pt /content/master_implementation/data/tiny-imagenet-200/

## 6. Verify Setup

In [ ]:
import os
import sys

# Add to Python path
sys.path.insert(0, '/content/master_implementation')

# Test imports
from guided_diffusion.script_util import create_model_and_diffusion
from rcdm.dataset import RepresentationDataset
from rcdm.encoder import load_encoder

# Verify imports are callable
print(f"✓ guided_diffusion: {create_model_and_diffusion.__name__}")
print(f"✓ rcdm.dataset: {RepresentationDataset.__name__}")
print(f"✓ rcdm.encoder: {load_encoder.__name__}")

# Check data file exists
data_path = '/content/master_implementation/data/tiny-imagenet-200/train_reps.pt'
assert os.path.exists(data_path), f"Data file not found: {data_path}"

print("\n✓ All imports successful")
print(f"✓ Data file found: {os.path.getsize(data_path) / 1e9:.2f} GB")

## 7. Start Training

Run your training script with GPU acceleration!

In [ ]:
# Training with GPU
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --device cuda

## 8. Monitor Training (Optional - Run in separate cell)

In [ ]:
# Check latest checkpoint
!ls -lh /content/drive/MyDrive/rcdm_checkpoints/

# Monitor GPU usage
!nvidia-smi

## 9. Generate Samples (After Training)

In [ ]:
# Load your sampling script here once you create it
# !python scripts/sampling.py --checkpoint /content/drive/MyDrive/rcdm_checkpoints/model_100000.pt

---

## Tips:

1. **Free GPU Runtime Limits**: Colab free tier has ~12 hour session limits. Save checkpoints frequently!
2. **Batch Size**: Increase `--batch_size` to 64-128 with GPU (you had 16 for CPU)
3. **Checkpoints**: Saved to Google Drive so they persist after session ends
4. **Monitoring**: Run `!nvidia-smi` periodically to check GPU utilization
5. **Interruptions**: If disconnected, re-run cells and use `--resume` flag with your latest checkpoint

## Resume Training:
```bash
!python scripts/train.py \
    --resume /content/drive/MyDrive/rcdm_checkpoints/model_25000.pt \
    --device cuda
```